In [1]:
import sys
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras as keras
sys.modules["tensorflow.keras"] = keras
sys.modules["keras"] = keras

import tensorflow as tf

from tf_keras.models import Sequential
from tf_keras.layers import MaxPooling2D, Activation, Flatten, Dropout

import qkeras

from qkeras import QDense, QConv2D
from qkeras.quantizers import quantized_bits

I0000 00:00:1777293410.326079  219881 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777293410.326502  219881 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777293410.358076  219881 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777293411.064900  219881 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

ModuleNotFoundError: No module named 'qkeras'

In [ ]:
import scipy.io
import numpy as np

def load_svhn(path):
    data = scipy.io.loadmat(path)
    X = data["X"]
    y = data["y"].reshape(-1)

    X = np.transpose(X, (3,0,1,2))

    y[y == 10] = 0

    return X, y

In [ ]:
X_train, y_train = load_svhn("train_32x32.mat")
X_test, y_test   = load_svhn("test_32x32.mat")

X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

In [ ]:
input_shape = (32, 32, 3)


X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print("X_train shape:", X_train.shape)

y_train[y_train == 10] = 0
y_test[y_test == 10] = 0


y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

X_train shape: (73257, 32, 32, 3)


In [ ]:
def build_model():
    model = Sequential()

    model.add(QConv2D(
        32, (3,3),
        padding="same",
        input_shape=(32,32,3),
        kernel_quantizer=quantized_bits(8,0,1),
        bias_quantizer=quantized_bits(8,0,1)
    ))
    model.add(Activation("relu")) 
    model.add(MaxPooling2D())

    model.add(QConv2D(
        64, (3,3),
        padding="same",
        kernel_quantizer=quantized_bits(8,0,1),
        bias_quantizer=quantized_bits(8,0,1)
    ))
    model.add(Activation("relu"))
    model.add(MaxPooling2D())

    model.add(Flatten())

    model.add(QDense(
        128,
        kernel_quantizer=quantized_bits(8,0,1),
        bias_quantizer=quantized_bits(8,0,1)
    ))
    model.add(Activation("relu"))

    model.add(QDense(
        10,
        kernel_quantizer=quantized_bits(8,0,1),
        bias_quantizer=quantized_bits(8,0,1)
    ))
    model.add(Activation("softmax"))

    return model

In [16]:
model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128
)

NameError: name 'Sequential' is not defined